##Build Results Fact
1. Read silver results table
2. Read silver sprints table
3. Add new column session_type with values RACE or SPRINT
4. UNION results and sprints
5. Derive additional columns
     - is_win -> Indicates that the driver own the race
     - is_podium -> Indicates that the driver scored a podium result (1, 2, 3)
     - has_points -> Indicates that the driver has scored points
6. Write the transformed data to gold fact_session_results table

In [0]:
%run ../00-common/01.environment-config

In [0]:
target_table = f"{catalog_name}.{gold_schema}.fact_session_results"

In [0]:
from pyspark.sql import functions as F

In [0]:
#Reading the source tables
results_df = (
    spark.table(f"{catalog_name}.{silver_schema}.results")
         .withColumn("session_type", F.lit("RACE"))
         .drop("race_name", "race_date", "ingestion_timestamp", "source_file")
)

sprints_df = (
    spark.table(f"{catalog_name}.{silver_schema}.results")
         .withColumn("session_type", F.lit("SPRINT"))
         .drop("race_name", "race_date", "ingestion_timestamp", "source_file")
)

In [0]:
#Union on the 2 tables (results_df and sprints_df)
results_sprints_df = results_df.unionByName(sprints_df)

In [0]:
# #Derive the columns is_win, is_podium, has_points
# is_win -> Indicates that the driver own the race
# is_podium -> Indicates that the driver scored a podium result (1, 2, 3)
# has_points -> Indicates that the driver has scored points

fact_session_results_df = (
    results_sprints_df
        .withColumn("is_win", F.col("final_position") == 1)
        .withColumn("is_podium", F.col("final_position").between(1, 3))
        .withColumn("has_points", F.col("points") > 0)
)

In [0]:
display(fact_session_results_df.filter("season = 2025"))

season,round,constructor_id,driver_id,grid_position,completed_laps,driver_number,points,final_position,final_position_text,status,session_type,is_win,is_podium,has_points
2025,1,alpine,doohan,14,0,7,0.0,19,R,Retired,RACE,false,false,false
2025,1,alpine,gasly,9,57,10,0.0,11,11,Finished,RACE,false,false,false
2025,1,aston_martin,alonso,12,32,14,0.0,17,R,Retired,RACE,false,false,false
2025,1,aston_martin,stroll,13,57,18,8.0,6,6,Finished,RACE,false,false,true
2025,1,ferrari,hamilton,8,57,44,1.0,10,10,Finished,RACE,false,false,true
2025,1,ferrari,leclerc,7,57,16,4.0,8,8,Finished,RACE,false,false,true
2025,1,haas,bearman,20,57,87,0.0,14,14,Finished,RACE,false,false,false
2025,1,haas,ocon,19,57,31,0.0,13,13,Finished,RACE,false,false,false
2025,1,mclaren,norris,1,57,4,25.0,1,1,Finished,RACE,true,true,true
2025,1,mclaren,piastri,2,57,81,2.0,9,9,Finished,RACE,false,false,true


In [0]:
#Write the fact_session_results_df to the target table
(
    fact_session_results_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
)

In [0]:
display(spark.table(target_table))

season,round,constructor_id,driver_id,grid_position,completed_laps,driver_number,points,final_position,final_position_text,status,session_type,is_win,is_podium,has_points
1950,1,alfa,fagioli,2,70,3,6.0,2,2,Finished,RACE,false,true,true
1950,1,alfa,fangio,3,62,1,0.0,12,R,Oil leak,RACE,false,false,false
1950,1,alfa,farina,1,70,2,9.0,1,1,Finished,RACE,true,true,true
1950,1,alfa,reg_parnell,4,70,4,4.0,3,3,Finished,RACE,false,true,true
1950,1,alta,crossley,17,43,24,0.0,16,R,Transmission,RACE,false,false,false
1950,1,alta,kelly,19,57,23,0.0,13,R,Not classified,RACE,false,false,false
1950,1,era,gerard,13,67,12,0.0,6,6,+3 Laps,RACE,false,false,false
1950,1,era,harrison,15,67,11,0.0,7,7,+3 Laps,RACE,false,false,false
1950,1,era,leslie_johnson,12,2,8,0.0,21,R,Supercharger,RACE,false,false,false
1950,1,era,peter_walker,10,5,9,0.0,20,R,Gearbox,RACE,false,false,false
